# Querulus final (prod only)

Только **prod**-модель: `config_prod.json` (CF+RG) → AutoMLManager (`retro=False`, `hp_tune=False`) → метрики → финэффект → MLflow → email → экспорт сервиса.

Сравнение parity/prod — в `example.ipynb`.

Конфиги и τ пишет **collect**. HPO/FS здесь не запускаются.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
OUTBOXML_ROOT = PROJECT_ROOT.parent.parent
for _p in (SRC, OUTBOXML_ROOT, PROJECT_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
print("PROJECT_ROOT", PROJECT_ROOT)


In [ ]:
import logging
import warnings

import pandas as pd
from IPython.display import display

from configs import config as querulus_outboxml_config
from querulus.naming import DEFAULT_HIVE_TABLE, MODEL_VERSION as MODEL_SEMVER
from querulus.training.example_pipeline import (
    export_prod_service_artifacts,
    fit_prod_models,
    load_example_dataset,
    load_example_thresholds,
    resolve_example_paths,
    run_prod_metrics,
    run_prod_plots_and_email,
    run_test_prod_fin_effect,
)

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.2f}".format
logging.basicConfig(level=logging.INFO, format="%(message)s")


In [ ]:
USE_SYNTHETIC = False
SEND_MAIL = False  # True — письмо через AutoML review + QuerulusEMailDSResult в plots
LOG_MLFLOW = True

HIVE_TABLE = DEFAULT_HIVE_TABLE
MODEL_VERSION = MODEL_SEMVER

paths = resolve_example_paths(PROJECT_ROOT, use_synthetic=USE_SYNTHETIC)
print("paths", paths)


## Данные и конфиги


In [ ]:
bundle = load_example_dataset(
    paths,
    hive_table=HIVE_TABLE,
    model_version=MODEL_VERSION,
)
df = bundle.df
built = bundle.built
periods = bundle.periods
CF_NAME = bundle.cf_name
RG_NAME = bundle.rg_name

print("df.shape", df.shape, "|", bundle.dataset_source)
print("prod config", built["prod_path"])
display(periods["table"])


## Prod fit (AutoMLManager)

`config_prod.json` → fit CF+RG, MLflow-артефакты (если `LOG_MLFLOW`). HPO/FS не запускаются.


In [ ]:
thresholds = load_example_thresholds(PROJECT_ROOT)
thr_prod = thresholds.prod
print(f"τ prod (τ-cal collect) = {thr_prod:.2f}")

models = fit_prod_models(
    bundle,
    external_config=querulus_outboxml_config,
    threshold=thr_prod,
    use_automl=True,
    send_mail=SEND_MAIL,
    log_mlflow=LOG_MLFLOW,
)
dsm_prod = models.dsm_cf_prod
print("models", list(dsm_prod.get_result()))


## Метрики prod


In [ ]:
run_prod_metrics(models, bundle, threshold=thr_prod)


## Финэффект на Test_prod


In [ ]:
test_prod = run_test_prod_fin_effect(models, bundle, thresholds=thresholds)
fe_prod = test_prod.prod
display(test_prod.compare_table)


## Plots / email (опционально)


In [ ]:
run_prod_plots_and_email(
    models,
    bundle,
    external_config=querulus_outboxml_config,
    send_email=SEND_MAIL,
)


## Экспорт для сервиса


In [ ]:
export_result = export_prod_service_artifacts(
    models,
    bundle,
    paths,
    thresholds=thresholds,
)
print(export_result.meta_path)
